In [6]:
import pandas as pd
import numpy as np

In [1]:
from preprocessing import load_and_prepare, remove_outliers
DATA = "../data"

In [2]:
data = load_and_prepare(folder=DATA)

gaps: 0.467->0.811 and 1.343->2.141 | cutoffs 0.639, 1.742 | dropped 532 of 38330
gaps: 0.466->0.807 and 1.338->2.140 | cutoffs 0.637, 1.739 | dropped 677 of 48000
split cutoff: 2025-08-31
train       (37798, 13)
test        (9670, 13)
full        (47323, 13)
validation  (12000, 13)
chart       (31, 13)


In [3]:
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]
X_full, y_full = data["X_full"], data["y_full"]
X_val, X_chart = data["X_val"], data["X_chart"]
train, test = data["train"], data["test"]

In [4]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

clean_idx = remove_outliers(test, verbose=False).index

def evaluate(name, pred):
    pred = pd.Series(pred, index=X_test.index)
    mae_all = mean_absolute_error(y_test, pred)
    mae_clean = mean_absolute_error(y_test[clean_idx], pred[clean_idx])
    rmse_clean = root_mean_squared_error(y_test[clean_idx], pred[clean_idx])
    print(f"{name:18s} | MAE all ${mae_all:7.1f} | MAE clean ${mae_clean:6.1f} | RMSE clean ${rmse_clean:6.1f}")

In [7]:
BANDS = [0, 300, 800, 1500, 4000]
rpm = train["posted_rate"] / train["distance"]
typical = rpm.groupby([train["equipment"], pd.cut(train["distance"], BANDS)], observed=True).median()

keys = pd.MultiIndex.from_arrays([test["equipment"], pd.cut(test["distance"], BANDS)])
evaluate("baseline", typical.reindex(keys).values * test["distance"].values)

baseline           | MAE all $  147.1 | MAE clean $  92.0 | RMSE clean $ 126.3


In [8]:
from sklearn.ensemble import HistGradientBoostingRegressor

params = dict(max_iter=500, learning_rate=0.05, early_stopping=False, random_state=42)

model = HistGradientBoostingRegressor(**params)
model.fit(X_train, y_train)
evaluate("gradient boosting", model.predict(X_test))

gradient boosting  | MAE all $  114.1 | MAE clean $  58.8 | RMSE clean $  84.9


In [10]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 3.7 MB/s  0:00:15m0:00:0100:01


In [11]:
from xgboost import XGBRegressor

xgb_params = dict(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,               # each tree sees a random 80% of rows
    colsample_bytree=0.8,        # and a random 80% of features
    objective="reg:absoluteerror",
    random_state=42,
    n_jobs=-1,
)

xgb = XGBRegressor(**xgb_params)
xgb.fit(X_train, y_train)
evaluate("xgboost", xgb.predict(X_test))

xgboost            | MAE all $  112.2 | MAE clean $  56.8 | RMSE clean $  84.0


# Stage 2 with the full dataset